## Install dependencies

In [1]:
import torch, sys, subprocess
print("torch:", torch.__version__, "| python:", sys.version.split()[0], "| cuda:", torch.cuda.is_available())
TORCH = torch.__version__.split("+")[0]
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))
pip("torch-geometric==2.6.1", "torch-scatter", "torch-sparse", "torch-cluster",
    "-f", f"https://data.pyg.org/whl/torch-{TORCH}+cpu.html")
pip("pyyaml", "scipy", "pandas", "matplotlib", "seaborn")
print("deps OK")

torch: 2.10.0+cu128 | python: 3.12.13 | cuda: True
deps OK


In [2]:
import os
import subprocess

REPO_DIR = "/kaggle/working/cosmic-net"
REPO_URL = "https://github.com/Rusheel86/cosmic-net.git"

# Read GitHub PAT from Kaggle Secrets
# If using kaggle_secrets:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_PAT = user_secrets.get_secret("GITHUB_PAT")

# Clone only if repo doesn't already exist
if not os.path.exists(REPO_DIR):
    auth_url = REPO_URL.replace(
        "https://",
        f"https://x-access-token:{GITHUB_PAT}@"
    )

    # SECURITY: never let the token reach the notebook output. A failed
    # subprocess.check_call raises CalledProcessError whose message embeds the
    # full command line (i.e. the token) — re-raise a sanitized error instead.
    try:
        subprocess.check_call([
            "git", "clone",
            auth_url,
            REPO_DIR
        ])
    except subprocess.CalledProcessError:
        raise RuntimeError(
            f"git clone failed for {REPO_URL} (token redacted) — check the PAT "
            "secret, the repo URL, and Kaggle internet access."
        ) from None
    # The clone writes the token into .git/config (remote origin URL) — remove it.
    subprocess.check_call(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])

os.chdir(REPO_DIR)
print(os.listdir("."))

['explain', 'main.py', 'pytest.ini', '.gitattributes', 'scripts', 'rl.md', 'data', 'model', '.env.example', '.git', 'conftest.py', 'graph', 'frontend', 'rl-kaggle-notebooks.md', 'REPOWISE.md', 'notebooks', 'backend', 'README.md', 'requirements.txt', 'symbolic', 'raw_hdf5', 'docker-compose.yml', 'tests', 'training', 'LICENSE', '.gitignore', 'config', 'rl-lit-review.md', 'outputs', 'rls', 'kaggle', 'Jenkinsfile', 'deploy', 'rl-implementation-plan.md']


In [3]:
# Clone repo + mount uploaded data
import subprocess, os, shutil
if not os.path.exists("/kaggle/working/cosmic-net"):
    subprocess.check_call(["git", "clone", "https://github.com/Rusheel86/cosmic-net.git",
                           "/kaggle/working/cosmic-net"])
os.chdir("/kaggle/working/cosmic-net")
print(os.listdir("."))
# Upload tng100_clustered.csv + best_model_augmented.pt as a Kaggle dataset named "cosmicnet-data"
INPUT = "/kaggle/input/datasets/nealsalian/cosmicnet-data"
os.makedirs("data/raw", exist_ok=True)
shutil.copy(f"{INPUT}/tng100_clustered.csv", "data/raw/tng100_clustered.csv")
os.makedirs("kaggle", exist_ok=True)
if os.path.exists(f"{INPUT}/best_model_augmented.pt"):
    shutil.copy(f"{INPUT}/best_model_augmented.pt", "kaggle/best_model_augmented.pt")
print("data staged:", os.path.getsize("data/raw/tng100_clustered.csv")/1e6, "MB")

['explain', 'main.py', 'pytest.ini', '.gitattributes', 'scripts', 'rl.md', 'data', 'model', '.env.example', '.git', 'conftest.py', 'graph', 'frontend', 'rl-kaggle-notebooks.md', 'REPOWISE.md', 'notebooks', 'backend', 'README.md', 'requirements.txt', 'symbolic', 'raw_hdf5', 'docker-compose.yml', 'tests', 'training', 'LICENSE', '.gitignore', 'config', 'rl-lit-review.md', 'outputs', 'rls', 'kaggle', 'Jenkinsfile', 'deploy', 'rl-implementation-plan.md']
data staged: 1.068424 MB


In [4]:
#Load config, force CPU-safe worker settings
import sys, yaml, torch, numpy as np
sys.path.insert(0, ".")
with open("config/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["source"] = "tng"
cfg["data"]["num_workers"] = 0         
cfg["data"]["batch_size"] = 16
cfg["model"]["mc_samples"] = 30
cfg["rls"] = {
    "policy_hidden": 64, "lr": 0.001, "entropy_coef": 0.01, "value_coef": 0.5,
    "epochs": 60, "batch_size": 32,
    "target_sparsity_start": 0.9, "target_sparsity_end": 0.4,
    "sparsity_anneal_epochs": 40,
    "w_acc": 1.0, "w_sp": 0.5, "w_conn": 1.0, "w_virial": 1.0, "w_unc": 0.5,
    "virial_anneal_start_epoch": 10, "min_keep_frac": 0.1, "seed": 42,
    # TTA ("RL at inference") — tune on VAL split only, then freeze
    "tta_lr": 1e-4, "tta_steps": 10, "tta_mc_samples": 15, "tta_patience": 3,
    "tta_target_sparsity": 0.5,
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| source:", cfg["data"]["source"])

device: cuda | source: tng


In [5]:
print(cfg)

{'api': {'host': '0.0.0.0', 'port': 8000}, 'data': {'batch_size': 16, 'grouping': 'fof', 'num_workers': 0, 'source': 'tng', 'test_ratio': 0.15, 'tng': {'cache_dir': 'data/raw/tng_cache', 'clustered_file': 'data/raw/tng100_clustered.csv', 'min_subhalos_per_halo': 3, 'n_halos': 600, 'snapshot': 99}, 'train_ratio': 0.7, 'val_ratio': 0.15}, 'explain': {'gnnexplainer': {'epochs': 200, 'lr': 0.01, 'num_hops': 3}, 'method': 'pgexplainer', 'output_dir': 'outputs/explanations', 'pgexplainer': {'epochs': 30, 'lr': 0.003, 'num_hops': 3}}, 'graph': {'edge_features': ['distance', 'delta_v', 'cos_theta', 'mass_ratio', 'proj_sep'], 'hierarchical': False, 'k_neighbors': 8, 'method': 'radius', 'radius_mpc': 2.0, 'self_loops': True}, 'logging': {'format': '%(asctime)s - %(name)s - %(levelname)s - %(message)s', 'level': 'INFO', 'log_file': 'outputs/logs/cosmic_net.log'}, 'model': {'activation': 'leaky_relu', 'dropout': 0.05, 'edge_features': 5, 'hidden_dim': 64, 'mc_dropout': True, 'mc_samples': 30, 'nod

In [6]:
#Load halos, build graphs, split
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder, build_dataloaders
loader = get_loader(cfg)
halos = loader.load()
print("total halos:", len(halos), "| split", loader.split_data.__name__ if False else "")
train_halos, val_halos, test_halos = loader.split_data()
print(f"train={len(train_halos)} val={len(val_halos)} test={len(test_halos)}")
torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])   # AFTER split_data (it resets RNG)
train_loader, val_loader, test_loader = build_dataloaders(cfg, train_halos, val_halos, test_halos)
b0 = next(iter(test_loader))
print("graph:", b0.x.shape, b0.edge_index.shape, b0.edge_attr.shape, "y:", b0.y.shape)

TNG_API_KEY not set. Will try local files first.


total halos: 541 | split 
train=378 val=81 test=82
graph: torch.Size([160, 4]) torch.Size([2, 1450]) torch.Size([1450, 5]) y: torch.Size([16])


In [7]:
print(cfg)

!git log --oneline --all -20

!grep -R "hidden_dim" -n . --exclude-dir=.git | head -50

{'api': {'host': '0.0.0.0', 'port': 8000}, 'data': {'batch_size': 16, 'grouping': 'fof', 'num_workers': 0, 'source': 'tng', 'test_ratio': 0.15, 'tng': {'cache_dir': 'data/raw/tng_cache', 'clustered_file': 'data/raw/tng100_clustered.csv', 'min_subhalos_per_halo': 3, 'n_halos': 600, 'snapshot': 99}, 'train_ratio': 0.7, 'val_ratio': 0.15}, 'explain': {'gnnexplainer': {'epochs': 200, 'lr': 0.01, 'num_hops': 3}, 'method': 'pgexplainer', 'output_dir': 'outputs/explanations', 'pgexplainer': {'epochs': 30, 'lr': 0.003, 'num_hops': 3}}, 'graph': {'edge_features': ['distance', 'delta_v', 'cos_theta', 'mass_ratio', 'proj_sep'], 'hierarchical': False, 'k_neighbors': 8, 'method': 'radius', 'radius_mpc': 2.0, 'self_loops': True}, 'logging': {'format': '%(asctime)s - %(name)s - %(levelname)s - %(message)s', 'level': 'INFO', 'log_file': 'outputs/logs/cosmic_net.log'}, 'model': {'activation': 'leaky_relu', 'dropout': 0.05, 'edge_features': 5, 'hidden_dim': 64, 'mc_dropout': True, 'mc_samples': 30, 'nod

In [8]:
# Load frozen backbone and verify it predicts
from model.model import load_model
import torch
import torch_geometric.data as pyg_data

ckpt = "/kaggle/input/datasets/nealsalian/cosmicnet-data/best_model_augmented.pt"

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------
gnn = load_model(ckpt, cfg, device)
gnn.eval()

model_device = next(gnn.parameters()).device

# ---------------------------------------------------------
# Create a valid test graph
# Use the complete graph instead of taking 2 nodes
# with arbitrary edges.
# ---------------------------------------------------------
single = pyg_data.Data(
    x=b0.x,
    edge_index=b0.edge_index,
    edge_attr=b0.edge_attr
)

# One graph containing all nodes
single.batch = torch.zeros(
    single.x.shape[0],
    dtype=torch.long
)

# Move graph to the same device as the model
single = single.to(model_device)

# ---------------------------------------------------------
# Forward pass
# ---------------------------------------------------------
with torch.no_grad():
    pred, _ = gnn(single)

print(
    "smoke prediction:", pred.item(),
    "| device:", model_device,
    "| params:", sum(p.numel() for p in gnn.parameters())
)

smoke prediction: 13.024879455566406 | device: cuda:0 | params: 831809


In [10]:
# Inline RL helpers (policy net, sparsify, reward) — matches rls/ package
# PREFER: from rls.policy import EdgePolicyNet
# PREFER: from rls.sparsify import hard_mask, repair_connectivity
# PREFER: from rls.policy_gradient import bernoulli_logp
# (swap in the above if rls/ is importable — see TIP at the top of this file)
import torch
import torch.nn as nn
import torch.nn.functional as F

class EdgePolicyNet(nn.Module):
    def __init__(self, edge_dim=5, node_emb_dim=128, hidden_dim=64):
        super().__init__()
        self.node_proj = nn.Linear(node_emb_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)
        self.fc = nn.Sequential(nn.Linear(hidden_dim*4, hidden_dim), nn.LeakyReLU(0.1),
                                nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU(0.1),
                                nn.Linear(hidden_dim, 1))
    def forward(self, edge_attr, node_emb, edge_index, context):
        e = F.leaky_relu(self.edge_proj(edge_attr), 0.1)
        u, v = edge_index
        nu = F.leaky_relu(self.node_proj(node_emb[u]), 0.1)
        nv = F.leaky_relu(self.node_proj(node_emb[v]), 0.1)
        c = context.unsqueeze(0).expand(e.size(0), -1)
        return self.fc(torch.cat([nu, nv, e, c], dim=-1))

def hard_mask(probs, min_keep_frac=0.1):
    # BOOL mask: edge_index[:, mask] / edge_attr[mask] require bool; an int
    # 0/1 mask silently positional-indexes (duplicates edges 0 and 1).
    mask = (probs >= 0.5)
    k = int(torch.ceil(torch.tensor(min_keep_frac) * probs.numel()))
    if mask.sum() < k:
        mask = torch.zeros_like(mask, dtype=torch.bool)
        mask[torch.topk(probs, k).indices] = True
    return mask

def repair_connectivity(edge_index, mask):
    mask = mask.clone()
    kept = mask.bool()
    incident = torch.zeros(edge_index.max().item()+1, dtype=torch.long)
    for i in range(edge_index.shape[1]):
        if kept[i]:
            incident[edge_index[0, i]] += 1; incident[edge_index[1, i]] += 1
    for node in (incident == 0).nonzero(as_tuple=True)[0].tolist():
        cand = (edge_index == node).sum(dim=0).bool()
        if cand.any():
            mask[cand.nonzero(as_tuple=True)[0][0]] = True
    return mask

def bernoulli_logp(p, action, eps=1e-8):
    return torch.where(action.bool(), torch.log(p.clamp(eps, 1.0)),
                       torch.log((1-p).clamp(eps, 1.0)))

def virial_ratio_pruned(ke, pe, eps=1e-8):
    return (2.0 * ke) / torch.abs(pe).clamp(min=eps)

In [20]:
# Baseline masks (random / degree / distance / mass-ratio / gradient-saliency /
# attention-topk / gumbel). PREFER: from rls.baselines import * (see TIP at top of file).
def random_mask(edge_index, edge_attr, pos, frac):
    device = edge_index.device
    n = edge_index.shape[1]

    k = max(1, int(n * frac))

    perm = torch.randperm(n, device=device)

    mask = torch.zeros(
        n,
        dtype=torch.bool,
        device=device
    )

    mask[perm[:k]] = True

    return mask

def degree_mask(edge_index, edge_attr, pos, frac):
    device = edge_index.device

    n_edges = edge_index.shape[1]

    # Number of nodes
    num_nodes = int(edge_index.max().item()) + 1

    deg = torch.zeros(
        num_nodes,
        device=device,
        dtype=torch.float32
    )

    ones = torch.ones(
        n_edges,
        device=device,
        dtype=torch.float32
    )

    deg.index_add_(0, edge_index[0], ones)
    deg.index_add_(0, edge_index[1], ones)

    edge_degree = (
        deg[edge_index[0]] +
        deg[edge_index[1]]
    ) / 2.0

    k = max(1, int(n_edges * frac))

    _, indices = torch.topk(
        edge_degree,
        k=min(k, n_edges)
    )

    mask = torch.zeros(
        n_edges,
        dtype=torch.bool,
        device=device
    )

    mask[indices] = True

    return mask
       

def distance_mask(edge_index, edge_attr, pos, frac):
    device = edge_index.device
    n = edge_index.shape[1]

    src = edge_index[0]
    dst = edge_index[1]

    distances = torch.norm(
        pos[src] - pos[dst],
        dim=1
    )

    k = max(1, int(n * frac))

    # Keep shortest-distance edges
    _, indices = torch.topk(
        distances,
        k=min(k, n),
        largest=False
    )

    mask = torch.zeros(
        n,
        dtype=torch.bool,
        device=device
    )

    mask[indices] = True

    return mask

def mass_ratio_mask(edge_index, edge_attr, pos, frac):
    device = edge_index.device
    n = edge_index.shape[1]

    # mass_ratio is edge feature index 3
    scores = edge_attr[:, 3]

    k = max(1, int(n * frac))

    _, indices = torch.topk(
        scores,
        k=min(k, n),
        largest=True
    )

    mask = torch.zeros(
        n,
        dtype=torch.bool,
        device=device
    )

    mask[indices] = True

    return mask

def gradient_saliency_mask(edge_index, edge_attr, pos, frac, graph=None):
    # The repo's existing explainer pathway (explain/explainer.py), used as the
    # mandatory 'why RL?' control: gradient importance on node features.
    n = edge_index.shape[1]
    g = graph
    x = g.x.clone().requires_grad_(True)
    d = pg.Data(x=x, edge_index=g.edge_index, edge_attr=g.edge_attr)
    d.batch = torch.zeros(x.shape[0], dtype=torch.long, device=device)
    pred, _ = gnn(d)
    pred.backward()
    ng = x.grad.abs().sum(dim=1)
    scores = (ng[edge_index[0]] + ng[edge_index[1]]) / 2
    m = torch.zeros(n, dtype=torch.bool)
    m[torch.topk(scores, int(frac*n)).indices] = True
    return m

def attention_topk_mask(edge_index, edge_attr, pos, frac, scores):
    m = torch.zeros(edge_index.shape[1], dtype=torch.bool)
    m[torch.topk(scores, int(frac*edge_index.shape[1])).indices] = True
    return m

# FIX: attention_topk_mask needs `scores` from somewhere — previously nothing in this
# notebook produced them, so the baseline was defined but never actually runnable.
# We use a small, FIXED-SEED, UNTRAINED MLP over edge_attr as a stand-in "attention"
# scorer (a common, cheap proxy for this kind of control — it isn't meant to be a
# strong baseline, just a non-degenerate one RL should clearly beat). If your GNN
# backbone has real learned attention weights available, swap this out for those.
import torch
import torch.nn as nn

# Use the same device as the GNN
model_device = next(gnn.parameters()).device

class _AttentionScorer(nn.Module):
    def __init__(self, edge_dim, seed=42):
        super().__init__()

        torch.manual_seed(seed)

        self.net = nn.Sequential(
            nn.Linear(edge_dim, 64),
            nn.LeakyReLU(),
            nn.Linear(64, 32),
            nn.LeakyReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, edge_attr):
        edge_attr = edge_attr.to(next(self.parameters()).device)

        with torch.no_grad():
            return self.net(edge_attr).squeeze(-1)


attn_scorer = _AttentionScorer(
    edge_dim=len(cfg["graph"]["edge_features"]),
    seed=cfg["rls"]["seed"]
).to(model_device)

attn_scorer.eval()

print("GNN device:", model_device)
print("Attention scorer device:", next(attn_scorer.parameters()).device)

class GumbelEdgeMask(nn.Module):
    """Differentiable edge-mask control (the 'why not Gumbel?' baseline) — matches
    rls/baselines.py. Trained end-to-end with an UNFROZEN copy of the GNN: this is
    the direct differentiable alternative to the RL policy, and per the plan (Part 3)
    it's the one baseline that gets an unfair advantage (joint training) — say so
    explicitly if these numbers go in the paper.

    forward(hard=False): returns a continuous [0,1] "keep probability" per edge
    (Gumbel-Softmax relaxed) — used during TRAINING to scale edge_attr so the whole
    pipeline stays differentiable without needing to know the GNN's internals.
    forward(hard=True): straight-through discretized mask — used at EVAL time to
    build an actually-pruned graph (edge_index[:, mask]), same convention as every
    other baseline/the RL policy, so keep_frac and RMSE are directly comparable.
    """
    def __init__(self, edge_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(edge_dim, hidden_dim), nn.LeakyReLU(0.1),
                                 nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU(0.1),
                                 nn.Linear(hidden_dim, 1))
    def forward(self, edge_attr, hard=False, tau=0.5):
        logits = self.net(edge_attr).squeeze(-1)              # [E]
        logits = torch.stack([-logits, logits], dim=-1)        # [E, 2] drop/keep
        u = torch.rand_like(logits) + 1e-8
        g = -torch.log(-torch.log(u))
        soft = torch.softmax((logits + g) / tau, dim=-1)
        if hard:
            idx = soft.argmax(dim=-1)
            ste = (F.one_hot(idx, 2).float() - soft).detach() + soft
            return ste[:, 1]
        return soft[:, 1]

GNN device: cuda:0
Attention scorer device: cuda:0


In [21]:
print("=== DEVICE CHECK ===")
print("GNN:", next(gnn.parameters()).device)
print("Test graph x:", g.x.device)
print("Test graph edge_index:", g.edge_index.device)
print("Test graph edge_attr:", g.edge_attr.device)
print("Attention scorer:", next(attn_scorer.parameters()).device)

=== DEVICE CHECK ===
GNN: cuda:0
Test graph x: cuda:0
Test graph edge_index: cuda:0
Test graph edge_attr: cuda:0
Attention scorer: cuda:0


In [22]:
b = next(iter(test_loader))
b = b.to(model_device)

g = b.get_example(0)

pos = g.pos if hasattr(g, "pos") else g.x[:, :3]

mask = attention_topk_mask(
    g.edge_index,
    g.edge_attr,
    pos,
    0.5,
    attn_scorer(g.edge_attr)
)

print("mask device:", mask.device)
print("edges:", g.edge_index.shape[1])
print("kept:", mask.sum().item())

pred = pred_for_mask(g, mask)

print("Test prediction:", pred)

mask device: cpu
edges: 84
kept: 42
Test prediction: 12.485469818115234


In [23]:
# Evaluate baseline sparsifiers on the test set -> Pareto curve

import numpy as np
import pandas as pd
import torch
import torch_geometric.data as pg
from model.physics_loss import MetricsComputer


def pred_for_mask(graph, mask):
    model_device = next(gnn.parameters()).device

    # Make sure mask is on the same device as graph
    mask = mask.to(graph.edge_index.device)

    g = pg.Data(
        x=graph.x,
        edge_index=graph.edge_index[:, mask],
        edge_attr=graph.edge_attr[mask]
    )

    # IMPORTANT: batch must be on the same device as x
    g.batch = torch.zeros(
        g.x.shape[0],
        dtype=torch.long,
        device=g.x.device
    )

    # Make absolutely sure the complete graph is on model device
    g = g.to(model_device)

    with torch.no_grad():
        pred, _ = gnn(g)

    return pred.squeeze().item()


results = []

fractions = [0.1, 0.25, 0.4, 0.6, 0.8, 1.0]

methods = [
    ("random", random_mask),
    ("degree", degree_mask),
    ("distance", distance_mask),
    ("mass_ratio", mass_ratio_mask),
    ("grad_saliency", gradient_saliency_mask),
    ("attention_topk", attention_topk_mask),
]


for frac in fractions:

    for name, mask_fn in methods:

        preds = []
        targets_current = []
        keeps = []

        for b in test_loader:

            # Move batch to model device
            b = b.to(device)

            for i in range(b.num_graphs):

                g = b.get_example(i)

                pos = (
                    g.pos
                    if hasattr(g, "pos")
                    else g.x[:, :3]
                )

                # Generate edge mask
                if name == "grad_saliency":

                    mask = mask_fn(
                        g.edge_index,
                        g.edge_attr,
                        pos,
                        frac,
                        graph=g
                    )

                elif name == "attention_topk":

                    mask = mask_fn(
                        g.edge_index,
                        g.edge_attr,
                        pos,
                        frac,
                        attn_scorer(g.edge_attr)
                    )

                else:

                    mask = mask_fn(
                        g.edge_index,
                        g.edge_attr,
                        pos,
                        frac
                    )

                # Prediction
                preds.append(
                    pred_for_mask(g, mask)
                )

                # Target
                targets_current.append(
                    g.y.item()
                )

                # Fraction of edges retained
                keeps.append(
                    mask.float().mean().item()
                )

        preds_tensor = torch.tensor(
            preds,
            dtype=torch.float32
        )

        targets_tensor = torch.tensor(
            targets_current,
            dtype=torch.float32
        )

        metrics = MetricsComputer.compute_all(
            preds_tensor,
            targets_tensor
        )

        results.append({
            "method": name,
            "frac": frac,
            "rmse": metrics["rmse"],
            "r2": metrics["r2"],
            "keep_frac": np.mean(keeps)
        })


# ---------------------------------------------------------
# Full graph reference
# ---------------------------------------------------------

full_preds = []
full_targets = []

for b in test_loader:

    b = b.to(device)

    for i in range(b.num_graphs):

        g = b.get_example(i)

        # Keep every edge
        full_mask = torch.ones(
            g.edge_index.shape[1],
            dtype=torch.bool,
            device=g.edge_index.device
        )

        full_preds.append(
            pred_for_mask(g, full_mask)
        )

        full_targets.append(
            g.y.item()
        )


full_preds_tensor = torch.tensor(
    full_preds,
    dtype=torch.float32
)

full_targets_tensor = torch.tensor(
    full_targets,
    dtype=torch.float32
)

full_m = MetricsComputer.compute_all(
    full_preds_tensor,
    full_targets_tensor
)

results.append({
    "method": "full",
    "frac": 1.0,
    "rmse": full_m["rmse"],
    "r2": full_m["r2"],
    "keep_frac": 1.0
})


# ---------------------------------------------------------
# Save results
# ---------------------------------------------------------

baseline_df = pd.DataFrame(results)

baseline_df.to_csv(
    "outputs/rls/baselines.csv",
    index=False
)

print(baseline_df.to_string(index=False))

print(
    "\nFull-graph reference RMSE:",
    round(full_m["rmse"], 4)
)

        method  frac     rmse         r2  keep_frac
        random  0.10 1.210010  -8.944613   0.098852
        degree  0.10 2.345601 -36.369583   0.098852
      distance  0.10 3.185808 -67.936501   0.098852
    mass_ratio  0.10 1.390257 -12.128049   0.098852
 grad_saliency  0.10 1.612748 -16.666203   0.098852
attention_topk  0.10 3.177374 -67.571976   0.098852
        random  0.25 0.407106  -0.125706   0.249226
        degree  0.25 1.960780 -25.113668   0.249226
      distance  0.25 1.252303  -9.651942   0.249226
    mass_ratio  0.25 0.722296  -2.543566   0.249226
 grad_saliency  0.25 0.805103  -3.402643   0.249226
attention_topk  0.25 1.266419  -9.893445   0.249226
        random  0.40 0.151256   0.844605   0.398815
        degree  0.40 1.403794 -12.384948   0.398815
      distance  0.40 0.667051  -2.022233   0.398815
    mass_ratio  0.40 0.401087  -0.092667   0.398815
 grad_saliency  0.40 0.456740  -0.416929   0.398815
attention_topk  0.40 0.676679  -2.110107   0.398815
        rand

In [24]:
# Save baselines plot
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4.5))
for name, grp in baseline_df[baseline_df.method != "full"].groupby("method"):
    plt.plot(grp.keep_frac, grp.rmse, "o-", label=name)
plt.axhline(full_m["rmse"], color="k", ls="--", label="full graph")
plt.xlabel("mean keep fraction"); plt.ylabel("RMSE (dex)"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("outputs/rls/baselines_pareto.png", dpi=150, bbox_inches="tight")
print("saved outputs/rls/baselines_pareto.png")

saved outputs/rls/baselines_pareto.png


In [25]:
# Gumbel-softmax joint baseline — training (~10-15 min on T4, 20 epochs)
# FIX: this baseline was speced in the plan (Task 6) and called out as "the single
# most important" Phase-0 baseline ("why RL?" control) but was missing from this
# notebook entirely. Added here.
#
# Design: an UNFROZEN COPY of the GNN backbone is trained jointly with GumbelEdgeMask
# to predict halo mass while a sparsity penalty pushes the mean keep-probability
# toward gumbel_target_sparsity. During training the mask is kept SOFT (forward(hard=
# False)) and used to scale edge_attr — this keeps the whole thing differentiable
# without assuming anything about the GNN's internals (no edge_weight argument
# required). At eval time (Cell 9c) the mask is discretized (hard=True, straight-
# through) to build a genuinely pruned graph, same convention as every other
# baseline, so keep_frac/RMSE are directly comparable.
# PREFER: from rls.baselines import GumbelEdgeMask (class already defined in Cell 7
# above either way — this cell just trains it).
import copy as _copy

gumbel_gnn = _copy.deepcopy(gnn).to(device)
gumbel_gnn.train()
gumbel_mask_net = GumbelEdgeMask(edge_dim=len(cfg["graph"]["edge_features"]),
                                 hidden_dim=64).to(device)
g_opt = torch.optim.Adam(list(gumbel_gnn.parameters()) + list(gumbel_mask_net.parameters()),
                         lr=1e-4)
gumbel_target_sparsity = cfg["rls"]["target_sparsity_end"]  # match RL's converged sparsity
gumbel_epochs = 20
gumbel_w_sparsity = 1.0

def _gumbel_tau(epoch, start=1.0, end=0.3, total=gumbel_epochs):
    # anneal temperature: soft/exploratory early, closer to hard-decision later
    p = min(1.0, epoch / max(1, total))
    return start + (end - start) * p

gumbel_log = []
for epoch in range(gumbel_epochs):
    tau = _gumbel_tau(epoch)
    ep_losses, ep_keeps = [], []
    for b in train_loader:
        b = b.to(device)
        for i in range(b.num_graphs):
            g = b.get_example(i)
            p_keep = gumbel_mask_net(g.edge_attr, hard=False, tau=tau)   # [E], differentiable
            edge_attr_soft = g.edge_attr * p_keep.unsqueeze(-1)
            d = pg.Data(x=g.x, edge_index=g.edge_index, edge_attr=edge_attr_soft)
            d.batch = torch.zeros(g.x.shape[0], dtype=torch.long, device=device)
            pred, _ = gumbel_gnn(d)
            mse = F.mse_loss(pred.view(-1), g.y.view(-1).float())
            sp_pen = (p_keep.mean() - gumbel_target_sparsity) ** 2
            loss = mse + gumbel_w_sparsity * sp_pen
            g_opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(gumbel_gnn.parameters()) + list(gumbel_mask_net.parameters()), 1.0)
            g_opt.step()
            ep_losses.append(loss.item()); ep_keeps.append(p_keep.mean().item())
    gumbel_log.append([epoch, tau, float(np.mean(ep_losses)), float(np.mean(ep_keeps))])
    if epoch % 5 == 0:
        print(f"gumbel epoch {epoch}: tau={tau:.2f} loss={np.mean(ep_losses):.4f} "
              f"mean_keep_p={np.mean(ep_keeps):.2f} (target {gumbel_target_sparsity:.2f})")
torch.save(gumbel_mask_net.state_dict(), "outputs/rls/gumbel_mask_net.pt")
torch.save(gumbel_gnn.state_dict(), "outputs/rls/gumbel_gnn.pt")
pd.DataFrame(gumbel_log, columns=["epoch", "tau", "loss", "mean_keep_p"]).to_csv(
    "outputs/rls/gumbel_training_log.csv", index=False)
print("Gumbel joint training done.")

gumbel epoch 0: tau=1.00 loss=0.4411 mean_keep_p=0.38 (target 0.40)
gumbel epoch 5: tau=0.82 loss=0.3549 mean_keep_p=0.41 (target 0.40)
gumbel epoch 10: tau=0.65 loss=0.3988 mean_keep_p=0.36 (target 0.40)
gumbel epoch 15: tau=0.48 loss=0.3335 mean_keep_p=0.36 (target 0.40)
Gumbel joint training done.


In [26]:
# Gumbel-softmax joint baseline — evaluation (hard mask, single sparsity point)
# Reported as ONE row (not a 6-fraction sweep like the other baselines) because the
# sparsity is a training outcome here, not a free parameter — matches the paper
# table convention (plan Part 3: "Gumbel-softmax (joint)" gets one operating point).
# If you want a full Gumbel Pareto curve for the paper too, re-run Cells 9b/9c with
# different gumbel_target_sparsity values (extra ~10-15 min GPU time each).
gumbel_gnn.eval()
preds_g, targets_g, keeps_g = [], [], []
with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        for i in range(b.num_graphs):
            g = b.get_example(i)
            m = (gumbel_mask_net(g.edge_attr, hard=True) > 0.5)
            if m.sum() == 0:   # degenerate collapse guard, same floor as RL's hard_mask
                m = hard_mask(gumbel_mask_net(g.edge_attr, hard=False), cfg["rls"]["min_keep_frac"])
            m = repair_connectivity(g.edge_index, m)
            d = pg.Data(x=g.x, edge_index=g.edge_index[:, m], edge_attr=g.edge_attr[m])
            d.batch = torch.zeros(g.x.shape[0], dtype=torch.long, device=device)
            pred, _ = gumbel_gnn(d)
            preds_g.append(pred.item()); targets_g.append(g.y.item()); keeps_g.append(m.float().mean().item())
preds_g, targets_g = torch.tensor(preds_g), torch.tensor(targets_g)
gumbel_m = MetricsComputer.compute_all(preds_g, targets_g)
gumbel_row = {"method": "gumbel", "frac": float(np.mean(keeps_g)), "rmse": gumbel_m["rmse"],
              "r2": gumbel_m["r2"], "keep_frac": float(np.mean(keeps_g))}
print(gumbel_row)
# fold into baselines.csv so Notebook C's results table picks it up automatically
baseline_df = pd.concat([baseline_df, pd.DataFrame([gumbel_row])], ignore_index=True)
baseline_df.to_csv("outputs/rls/baselines.csv", index=False)
print("appended gumbel row to outputs/rls/baselines.csv")

{'method': 'gumbel', 'frac': 0.1814144962444538, 'rmse': 0.4437194764614105, 'r2': -0.3372945785522461, 'keep_frac': 0.1814144962444538}
appended gumbel row to outputs/rls/baselines.csv
